# 🤖 Anthropic API — Cuaderno de trabajo

Este cuaderno proporciona una interfaz organizada para interactuar con la API de Anthropic.  
Incluye funciones reutilizables para:
- Configurar la conexión y elegir el modelo
- Enviar mensajes simples y conversaciones
- Usar streaming
- Gestionar el historial de conversación
- Ejemplos prácticos de uso

---

## 1. Instalación de dependencias

In [1]:
# Instalar el SDK oficial de Anthropic
# %pip install anthropic --quiet

## 2. Importaciones

In [2]:
import os
import anthropic
from typing import Optional
from IPython.display import display, Markdown

## 3. Configuración — API Key y selección de modelo

Introduce tu API key y elige el modelo que quieres usar.

| Modelo | Descripción |
|--------|-------------|
| `claude-opus-4-5` | Máxima capacidad, ideal para tareas complejas |
| `claude-sonnet-4-5` | Balance óptimo entre velocidad y calidad |
| `claude-haiku-4-5` | Más rápido y económico, ideal para tareas sencillas |

In [3]:
# ── Configuración ────────────────────────────────────────────────────────────

# Opción A: escribe la key directamente (solo para uso local, nunca la subas a git)
# ANTHROPIC_API_KEY = "sk-ant-..."  # ← reemplaza con tu key

# Opción B: léela desde una variable de entorno (recomendado)
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Modelos disponibles
MODELOS_DISPONIBLES = [
    "claude-opus-4-5",
    "claude-sonnet-4-5",
    "claude-haiku-4-5",
]

# Elige el modelo (cambia el índice: 0=Opus, 1=Sonnet, 2=Haiku)
MODELO_ELEGIDO = MODELOS_DISPONIBLES[1]  # claude-sonnet-4-5 por defecto

print(f"✅ Modelo seleccionado: {MODELO_ELEGIDO}")

✅ Modelo seleccionado: claude-sonnet-4-5


## 4. Funciones principales

In [4]:
# ── 4.1  Crear cliente ────────────────────────────────────────────────────────

def crear_cliente(api_key: str) -> anthropic.Anthropic:
    """
    Crea y devuelve un cliente Anthropic autenticado.

    Args:
        api_key: Tu API key de Anthropic.

    Returns:
        Cliente Anthropic listo para usar.

    Raises:
        ValueError: Si la API key está vacía.
    """
    if not api_key or api_key.startswith("sk-ant-..."):
        raise ValueError(
            "API key inválida. Reemplaza el valor en la celda de Configuración."
        )
    return anthropic.Anthropic(api_key=api_key)


# ── 4.2  Consulta simple ──────────────────────────────────────────────────────

def consultar(
    cliente: anthropic.Anthropic,
    prompt: str,
    modelo: str = MODELO_ELEGIDO,
    sistema: Optional[str] = None,
    max_tokens: int = 1024,
    temperatura: float = 1.0,
) -> str:
    """
    Envía un mensaje al modelo y devuelve el texto de respuesta.

    Args:
        cliente:    Cliente Anthropic.
        prompt:     Mensaje del usuario.
        modelo:     Identificador del modelo a usar.
        sistema:    Prompt de sistema opcional (personalidad, contexto, etc.).
        max_tokens: Número máximo de tokens en la respuesta.
        temperatura: Creatividad de la respuesta (0.0–1.0).

    Returns:
        Texto de respuesta del modelo.
    """
    kwargs = {
        "model": modelo,
        "max_tokens": max_tokens,
        "temperature": temperatura,
        "messages": [{"role": "user", "content": prompt}],
    }
    if sistema:
        kwargs["system"] = sistema

    respuesta = cliente.messages.create(**kwargs)
    return respuesta.content[0].text


# ── 4.3  Conversación con historial ──────────────────────────────────────────

def iniciar_conversacion() -> list:
    """Devuelve un historial de conversación vacío."""
    return []


def conversar(
    cliente: anthropic.Anthropic,
    historial: list,
    mensaje: str,
    modelo: str = MODELO_ELEGIDO,
    sistema: Optional[str] = None,
    max_tokens: int = 1024,
) -> str:
    """
    Añade un turno a la conversación y devuelve la respuesta del modelo.
    Modifica `historial` en el lugar para mantener el contexto entre llamadas.

    Args:
        cliente:   Cliente Anthropic.
        historial: Lista de mensajes acumulados (se modifica in-place).
        mensaje:   Nuevo mensaje del usuario.
        modelo:    Modelo a usar.
        sistema:   Prompt de sistema opcional.
        max_tokens: Tokens máximos en la respuesta.

    Returns:
        Texto de la respuesta del asistente.
    """
    historial.append({"role": "user", "content": mensaje})

    kwargs = {
        "model": modelo,
        "max_tokens": max_tokens,
        "messages": historial,
    }
    if sistema:
        kwargs["system"] = sistema

    respuesta = cliente.messages.create(**kwargs)
    texto = respuesta.content[0].text

    historial.append({"role": "assistant", "content": texto})
    return texto


# ── 4.4  Streaming ────────────────────────────────────────────────────────────

def consultar_streaming(
    cliente: anthropic.Anthropic,
    prompt: str,
    modelo: str = MODELO_ELEGIDO,
    sistema: Optional[str] = None,
    max_tokens: int = 1024,
) -> str:
    """
    Envía un mensaje con streaming: imprime cada fragmento en tiempo real
    y devuelve el texto completo al finalizar.

    Args:
        cliente:   Cliente Anthropic.
        prompt:    Mensaje del usuario.
        modelo:    Modelo a usar.
        sistema:   Prompt de sistema opcional.
        max_tokens: Tokens máximos en la respuesta.

    Returns:
        Texto completo de la respuesta.
    """
    kwargs = {
        "model": modelo,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if sistema:
        kwargs["system"] = sistema

    texto_completo = []
    print("🤖 Respuesta (streaming):")
    with cliente.messages.stream(**kwargs) as stream:
        for fragmento in stream.text_stream:
            print(fragmento, end="", flush=True)
            texto_completo.append(fragmento)
    print()  # salto de línea al terminar
    return "".join(texto_completo)


# ── 4.5  Utilidades ───────────────────────────────────────────────────────────

def mostrar_respuesta(texto: str) -> None:
    """Renderiza la respuesta como Markdown en el notebook."""
    display(Markdown(texto))


def contar_tokens(
    cliente: anthropic.Anthropic,
    prompt: str,
    modelo: str = MODELO_ELEGIDO,
    sistema: Optional[str] = None,
) -> dict:
    """
    Estima el número de tokens de entrada sin enviar la petición.

    Returns:
        Diccionario con 'input_tokens'.
    """
    kwargs = {
        "model": modelo,
        "messages": [{"role": "user", "content": prompt}],
    }
    if sistema:
        kwargs["system"] = sistema

    resultado = cliente.messages.count_tokens(**kwargs)
    return {"input_tokens": resultado.input_tokens}


def info_uso(respuesta_raw) -> dict:
    """
    Extrae información de uso (tokens) de una respuesta raw de la API.

    Args:
        respuesta_raw: Objeto Message devuelto por cliente.messages.create().

    Returns:
        Diccionario con tokens de entrada y salida.
    """
    return {
        "input_tokens": respuesta_raw.usage.input_tokens,
        "output_tokens": respuesta_raw.usage.output_tokens,
    }


print("✅ Funciones cargadas correctamente.")

✅ Funciones cargadas correctamente.


## 5. Inicializar cliente

In [5]:
cliente = crear_cliente(ANTHROPIC_API_KEY)
print(f"✅ Cliente creado — modelo activo: {MODELO_ELEGIDO}")

✅ Cliente creado — modelo activo: claude-sonnet-4-5


---
## 6. Ejemplos de uso

### 6.1 Consulta simple

In [ ]:
respuesta = consultar(
    cliente,
    prompt="¿Cuáles son las tres leyes de la termodinámica? Explícalas brevemente.",
    sistema="Eres un profesor de física conciso y claro. Responde siempre en español.",
)

mostrar_respuesta(respuesta)

### 6.2 Consulta con temperatura personalizada

In [ ]:
respuesta_creativa = consultar(
    cliente,
    prompt="Escribe un haiku sobre la inteligencia artificial.",
    temperatura=1.0,   # más creatividad
    max_tokens=200,
)

mostrar_respuesta(respuesta_creativa)

### 6.3 Conversación multi-turno

In [ ]:
SISTEMA_TUTOR = "Eres un tutor de Python amigable. Explica con ejemplos de código breves."

historial = iniciar_conversacion()

# Turno 1
r1 = conversar(cliente, historial, "¿Qué es una list comprehension en Python?", sistema=SISTEMA_TUTOR)
print("👤 Turno 1")
mostrar_respuesta(r1)

# Turno 2  (el modelo recuerda el contexto anterior)
r2 = conversar(cliente, historial, "¿Puedes añadir un filtro condicional al ejemplo anterior?", sistema=SISTEMA_TUTOR)
print("👤 Turno 2")
mostrar_respuesta(r2)

print(f"\n📝 Turnos en el historial: {len(historial)}")

### 6.4 Respuesta con streaming

In [ ]:
texto_final = consultar_streaming(
    cliente,
    prompt="Dame 5 consejos para escribir código Python más limpio y mantenible.",
    sistema="Responde siempre en español con ejemplos prácticos.",
    max_tokens=800,
)

### 6.5 Conteo de tokens antes de enviar

In [ ]:
mi_prompt = "Analiza las ventajas y desventajas del lenguaje de programación Rust frente a C++."

tokens = contar_tokens(cliente, mi_prompt)
print(f"📊 Tokens estimados para el prompt: {tokens['input_tokens']}")

### 6.6 Cambiar el modelo en tiempo de ejecución

In [ ]:
# Puedes pasar cualquier modelo directamente a las funciones
modelo_rapido = "claude-haiku-4-5"

respuesta_haiku = consultar(
    cliente,
    prompt="Di 'Hola mundo' en 5 lenguajes de programación.",
    modelo=modelo_rapido,   # ← sobreescribe MODELO_ELEGIDO
    max_tokens=300,
)

print(f"Modelo usado: {modelo_rapido}")
mostrar_respuesta(respuesta_haiku)

---
## 7. Referencia rápida de funciones

| Función | Descripción |
|---------|-------------|
| `crear_cliente(api_key)` | Crea el cliente autenticado |
| `consultar(cliente, prompt, ...)` | Consulta simple, devuelve string |
| `iniciar_conversacion()` | Crea un historial vacío |
| `conversar(cliente, historial, mensaje, ...)` | Conversación multi-turno |
| `consultar_streaming(cliente, prompt, ...)` | Respuesta en tiempo real |
| `mostrar_respuesta(texto)` | Renderiza Markdown en el notebook |
| `contar_tokens(cliente, prompt, ...)` | Estima tokens sin consumir cuota |
| `info_uso(respuesta_raw)` | Extrae tokens usados de una respuesta raw |

---
> **Consejo:** Para entornos de producción, guarda tu API key en una variable de entorno (`ANTHROPIC_API_KEY`) y cárgala con `os.environ.get("ANTHROPIC_API_KEY")`.